# Session 7: Build MoodNet

Click a cell and press **Shift + Enter** to run it. Work from top to bottom.
Cells marked **YOUR TURN** are for you to finish. Run the first cell straight away: it draws 15,000 faces.

In [ ]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
import matplotlib.pyplot as plt
from emoji_faces import make_face, CLASSES

if torch.accelerator.is_available():
    device = torch.accelerator.current_accelerator()
else:
    device = torch.device("cpu")
print("Using", device)

class EmojiFaces(Dataset):
    def __init__(self, how_many, first_seed, transform=None):
        self.faces = []
        for seed in range(first_seed, first_seed + how_many):
            picture, mood = make_face(seed)
            if transform:
                picture = transform(picture)
            self.faces.append((picture, mood))

    def __len__(self):
        return len(self.faces)

    def __getitem__(self, idx):
        return self.faces[idx]

get_ready = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

print("Drawing faces... (about a minute)")
train_faces = EmojiFaces(12000, first_seed=0, transform=get_ready)
test_faces = EmojiFaces(3000, first_seed=1_000_000, transform=get_ready)
train_loader = DataLoader(train_faces, batch_size=64, shuffle=True)
test_loader = DataLoader(test_faces, batch_size=64)
print(len(train_faces), "training faces and", len(test_faces), "test faces")

## 1. Flatten

In [ ]:
faces, moods = next(iter(train_loader))
flat = nn.Flatten()(faces)
print("a batch of faces:", faces.shape, "-> flattened:", flat.shape)

## 2. MoodNet

In [ ]:
class MoodNet(nn.Module):
    def __init__(self, hidden=256):
        super().__init__()
        self.flatten = nn.Flatten()
        self.hidden1 = nn.Linear(3 * 32 * 32, hidden)
        self.hidden2 = nn.Linear(hidden, 128)
        self.output = nn.Linear(128, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.flatten(x)
        x = self.relu(self.hidden1(x))
        x = self.relu(self.hidden2(x))
        return self.output(x)

torch.manual_seed(1)      # the same random starting weights for everyone
model = MoodNet().to(device)
print(model)

In [ ]:
# YOUR TURN: work out MoodNet's weights and biases layer by layer, then check.
for name, p in model.named_parameters():
    print(name, tuple(p.shape), p.numel())
print("total:", ...)

## 3. A Forward Pass

In [ ]:
with torch.no_grad():
    scores = model(faces.to(device))
print("scores for the whole batch:", scores.shape)
print("10 scores for face 0:", [round(s, 2) for s in scores[0].tolist()])
chances = torch.softmax(scores[0], dim=0)
print("as chances:", [round(c, 3) for c in chances.tolist()])
print("chances add up to", round(chances.sum().item(), 3))
print("guess:", CLASSES[scores[0].argmax().item()], "  right answer:", CLASSES[moods[0]])

## 4. How Good Is It Before Training?

In [ ]:
correct = 0
with torch.no_grad():
    for f, m in test_loader:
        correct += (model(f.to(device)).argmax(dim=1) == m.to(device)).sum().item()
print("accuracy before training:", round(100 * correct / len(test_faces), 1), "%")